In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (

    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.optimizers import (
    Adam,
    RMSprop
)

In [20]:
data=pd.read_csv('/kaggle/input/datasets/manishavelamani/dataset/PJME_preprocessd.csv',parse_dates=['Datetime'],index_col='Datetime')

In [21]:
data.columns

Index(['PJME_MW', 'PJME_MW_Scaled', 'Hour', 'Day', 'Week', 'Month',
       'DayOfWeek', 'Weekend', 'Lag_1', 'Lag_24', 'Lag_48', 'Lag_168',
       'RollingMean_24', 'RollingStd_24', 'RollingMean_168'],
      dtype='object')

In [22]:
from sklearn.preprocessing import MinMaxScaler

features = [
    'PJME_MW_Scaled',
    'Hour',
    'Day',
    'Week',
    'Month',
    'DayOfWeek',
    'Weekend',
    'Lag_1',
    'Lag_24',
    'Lag_48',
    'Lag_168',
    'RollingMean_24',
    'RollingStd_24',
    'RollingMean_168'
]

target = 'PJME_MW_Scaled'

# Select input features
multivariate_data = data[features].copy()

# Scale ALL input features
feature_scaler = MinMaxScaler()

multivariate_data_scaled = pd.DataFrame(
    feature_scaler.fit_transform(multivariate_data),
    columns=features,
    index=multivariate_data.index
)

multivariate_data_scaled.head()

,PJME_MW_Scaled,Hour,Day,Week,Month,DayOfWeek,Weekend,Lag_1,Lag_24,Lag_48,Lag_168,RollingMean_24,RollingStd_24,RollingMean_168
Datetime,,,,,,,,,,,,,,
2002-01-08 01:00:00,0.433011,0.043478,0.233333,0.019231,0.0,0.166667,0.0,0.486937,0.353052,0.360420,0.462358,0.535733,0.407201,0.421362
2002-01-08 02:00:00,0.409021,0.086957,0.233333,0.019231,0.0,0.166667,0.0,0.433011,0.325625,0.329371,0.427439,0.539989,0.386780,0.421179
2002-01-08 03:00:00,0.399889,0.130435,0.233333,0.019231,0.0,0.166667,0.0,0.409021,0.315255,0.319960,0.399331,0.544309,0.363683,0.421185
2002-01-08 04:00:00,0.405058,0.173913,0.233333,0.019231,0.0,0.166667,0.0,0.399889,0.316029,0.315750,0.385154,0.548853,0.338064,0.421382
2002-01-08 05:00:00,0.427316,0.217391,0.233333,0.019231,0.0,0.166667,0.0,0.405058,0.336522,0.319496,0.390045,0.553487,0.312793,0.421752


In [23]:
train_size = int(len(multivariate_data_scaled) * 0.70)
val_size = int(len(multivariate_data_scaled) * 0.10)

train = multivariate_data_scaled.iloc[:train_size]
validation = multivariate_data_scaled.iloc[train_size:train_size + val_size]
test = multivariate_data_scaled.iloc[train_size + val_size:]

In [24]:
def create_multivariate_sequences(df, sequence_length, forecast_horizon=24, target_col='PJME_MW_Scaled'):
    X = []
    y = []

    values = df.values
    target_index = df.columns.get_loc(target_col)

    for i in range(len(df) - sequence_length - forecast_horizon + 1):
        X.append(values[i:i + sequence_length])

        y.append(
            values[
                i + sequence_length:
                i + sequence_length + forecast_horizon,
                target_index
            ]
        )

    return np.array(X), np.array(y)

In [25]:
sequence_length = 168

X_train, y_train = create_multivariate_sequences(train, sequence_length)
X_val, y_val = create_multivariate_sequences(validation, sequence_length)
X_test, y_test = create_multivariate_sequences(test, sequence_length)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)

print('X_val:', X_val.shape)
print('y_val:', y_val.shape)

print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train: (101465, 168, 14)
y_train: (101465, 24)
X_val: (14331, 168, 14)
y_val: (14331, 24)
X_test: (28855, 168, 14)
y_test: (28855, 24)


In [26]:
rnn_phase6 = Sequential([
    SimpleRNN(32, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

rnn_phase6.compile(
    optimizer='adam',
    loss='mse'
)

rnn_phase6.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_3 (SimpleRNN)        │ (None, 32)             │         1,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │           792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,296 (8.97 KB)

 Trainable params: 2,296 (8.97 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
baseline_history = rnn_phase6.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 40s 12ms/step - loss: 0.0133 - val_loss: 0.0050
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.0042 - val_loss: 0.0044
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.0035 - val_loss: 0.0039
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0033 - val_loss: 0.0037
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0032 - val_loss: 0.0037
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0031 - val_loss: 0.0038
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0030 - val_loss: 0.0040
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0030 - val_loss: 0.0036
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0029 - val_loss: 0.0038
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0029 - val_loss: 0.0036


In [28]:
from sklearn.preprocessing import MinMaxScaler
# Create scaler using the original MW values
scaler = MinMaxScaler()
scaler.fit(data[['PJME_MW']])

MinMaxScaler()

In [29]:
def evaluate_model(model, X_test, y_test, scaler, sequence_length, model_name):
    predictions = model.predict(X_test, verbose=0)
    # Convert scaled values back to original MW values
    pred_original = scaler.inverse_transform(predictions.reshape(-1, 1)).reshape(predictions.shape)
    y_original = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    mae = mean_absolute_error(y_original.flatten(),pred_original.flatten())
    mse = mean_squared_error(y_original.flatten(),pred_original.flatten())
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_original.flatten(),pred_original.flatten()) * 100
    r2 = r2_score(y_original.flatten(),pred_original.flatten())
    bias = np.mean(pred_original.flatten() - y_original.flatten())
    return {
        'Sequence Length': sequence_length,
        'Model': model_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2,
        'Bias': bias
    }

In [30]:
baseline_rnn_result = evaluate_model(
    rnn_phase6,
    X_test,
    y_test,
    scaler,
    168,
    'Baseline RNN (Multivariate)'
)

pd.DataFrame([baseline_rnn_result])

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,Baseline RNN (Multivariate),1420.956207,3.852737e+06,1962.838902,4.56125,0.903498,127.218473


# RNN + EarlyStopping

In [31]:
# EarlyStopping callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# RNN model
rnn_early = Sequential([
    SimpleRNN(32, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

rnn_early.compile(
    optimizer='adam',
    loss='mse'
)

rnn_early.summary()

# Train with EarlyStopping
rnn_early_history = rnn_early.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_4 (SimpleRNN)        │ (None, 32)             │         1,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 24)             │           792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,296 (8.97 KB)

 Trainable params: 2,296 (8.97 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 41s 12ms/step - loss: 0.0126 - val_loss: 0.0052
Epoch 2/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0044 - val_loss: 0.0045
Epoch 3/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0038 - val_loss: 0.0041
Epoch 4/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0035 - val_loss: 0.0040
Epoch 5/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0033 - val_loss: 0.0039
Epoch 6/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0032 - val_loss: 0.0039
Epoch 7/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.0032 - val_loss: 0.0039
Epoch 8/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0031 - val_loss: 0.0036
Epoch 9/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0031 - val_loss: 0.0038
Epoch 10/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0030 - val_loss: 0.0037
Epoch 11/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0030 - val_loss: 0.0035
Epoch 12

In [32]:
rnn_early_result = evaluate_model(
    rnn_early,
    X_test,
    y_test,
    scaler,
    168,
    'RNN + EarlyStopping'
)

pd.DataFrame([rnn_early_result])

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + EarlyStopping,1404.789758,3.805389e+06,1950.7406,4.47675,0.904684,70.623907


In [33]:
comparison_rnn = pd.DataFrame([
    baseline_rnn_result,
    rnn_early_result
])

comparison_rnn

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,Baseline RNN (Multivariate),1420.956207,3.852737e+06,1962.838902,4.56125,0.903498,127.218473
1,168,RNN + EarlyStopping,1404.789758,3.805389e+06,1950.740600,4.47675,0.904684,70.623907


# RNN+Dropout

In [34]:
rnn_dropout = Sequential([
    SimpleRNN(32, input_shape=(sequence_length, X_train.shape[2])),
    Dropout(0.2),
    Dense(24)
])

rnn_dropout.compile(
    optimizer='adam',
    loss='mse'
)

rnn_dropout.summary()

rnn_dropout_history = rnn_dropout.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_5 (SimpleRNN)        │ (None, 32)             │         1,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │           792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,296 (8.97 KB)

 Trainable params: 2,296 (8.97 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 41s 12ms/step - loss: 0.0215 - val_loss: 0.0051
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.0061 - val_loss: 0.0047
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0055 - val_loss: 0.0047
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0054 - val_loss: 0.0046
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0053 - val_loss: 0.0045


In [35]:
rnn_dropout_result = evaluate_model(
    rnn_dropout,
    X_test,
    y_test,
    scaler,
    168,
    'RNN + Dropout'
)

pd.DataFrame([rnn_dropout_result])

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + Dropout,1709.290569,5.318545e+06,2306.197167,5.450563,0.866782,-181.105101


In [36]:
comparison_rnn = pd.DataFrame([
    baseline_rnn_result,
    rnn_early_result,
    rnn_dropout_result
])

comparison_rnn

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,Baseline RNN (Multivariate),1420.956207,3.852737e+06,1962.838902,4.561250,0.903498,127.218473
1,168,RNN + EarlyStopping,1404.789758,3.805389e+06,1950.740600,4.476750,0.904684,70.623907
2,168,RNN + Dropout,1709.290569,5.318545e+06,2306.197167,5.450563,0.866782,-181.105101


# more units =64

In [37]:
rnn_more_units = Sequential([
    SimpleRNN(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

rnn_more_units.compile(
    optimizer='adam',
    loss='mse'
)

rnn_more_units.summary()

rnn_more_units_history = rnn_more_units.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_6 (SimpleRNN)        │ (None, 64)             │         5,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,616 (25.84 KB)

 Trainable params: 6,616 (25.84 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 41s 12ms/step - loss: 0.0083 - val_loss: 0.0049
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0040 - val_loss: 0.0041
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0034 - val_loss: 0.0038
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0032 - val_loss: 0.0037
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 39s 12ms/step - loss: 0.0031 - val_loss: 0.0040


In [38]:
rnn_more_units_result = evaluate_model(
    rnn_more_units,
    X_test,
    y_test,
    scaler,
    168,
    'RNN + More Units'
)

pd.DataFrame([rnn_more_units_result])

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + More Units,1640.79676,4.978140e+06,2231.174637,5.1999,0.875309,-435.698344


In [39]:
comparison_rnn = pd.DataFrame([
    baseline_rnn_result,
    rnn_early_result,
    rnn_dropout_result,
    rnn_more_units_result
])

comparison_rnn = comparison_rnn.sort_values('RMSE').reset_index(drop=True)

comparison_rnn

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + EarlyStopping,1404.789758,3.805389e+06,1950.740600,4.476750,0.904684,70.623907
1,168,Baseline RNN (Multivariate),1420.956207,3.852737e+06,1962.838902,4.561250,0.903498,127.218473
2,168,RNN + More Units,1640.796760,4.978140e+06,2231.174637,5.199900,0.875309,-435.698344
3,168,RNN + Dropout,1709.290569,5.318545e+06,2306.197167,5.450563,0.866782,-181.105101


# RNN (168) + Batch Normalization

In [40]:
rnn_batchnorm = Sequential([
    SimpleRNN(64, input_shape=(sequence_length, X_train.shape[2])),
    BatchNormalization(),
    Dense(24)
])

rnn_batchnorm.compile(
    optimizer='adam',
    loss='mse'
)

rnn_batchnorm.summary()

rnn_batchnorm_history = rnn_batchnorm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_7 (SimpleRNN)        │ (None, 64)             │         5,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,872 (26.84 KB)

 Trainable params: 6,744 (26.34 KB)

 Non-trainable params: 128 (512.00 B)

Epoch 1/50
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 42s 13ms/step - loss: 0.0230 - val_loss: 0.0165
Epoch 2/50
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0070 - val_loss: 0.0139
Epoch 3/50
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0057 - val_loss: 0.0067
Epoch 4/50
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0052 - val_loss: 0.0119
Epoch 5/50
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 39s 12ms/step - loss: 0.0051 - val_loss: 0.0063


In [41]:
rnn_batchnorm_result = evaluate_model(
    rnn_batchnorm,
    X_test,
    y_test,
    scaler,
    168,
    'RNN + BatchNormalization'
)

pd.DataFrame([rnn_batchnorm_result])

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + BatchNormalization,3246.8356,1.624687e+07,4030.740491,10.685205,0.593052,81.122771


In [42]:
comparison_rnn = pd.DataFrame([
    baseline_rnn_result,
    rnn_early_result,
    rnn_dropout_result,
    rnn_more_units_result,
    rnn_batchnorm_result
])

comparison_rnn = comparison_rnn.sort_values('RMSE').reset_index(drop=True)

comparison_rnn

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + EarlyStopping,1404.789758,3.805389e+06,1950.740600,4.476750,0.904684,70.623907
1,168,Baseline RNN (Multivariate),1420.956207,3.852737e+06,1962.838902,4.561250,0.903498,127.218473
2,168,RNN + More Units,1640.796760,4.978140e+06,2231.174637,5.199900,0.875309,-435.698344
3,168,RNN + Dropout,1709.290569,5.318545e+06,2306.197167,5.450563,0.866782,-181.105101
4,168,RNN + BatchNormalization,3246.835600,1.624687e+07,4030.740491,10.685205,0.593052,81.122771


# RNN + RMSprop

In [44]:
rnn_rmsprop = Sequential([
    SimpleRNN(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

rnn_rmsprop.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss='mse'
)

rnn_rmsprop_history = rnn_rmsprop.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - loss: 0.0085 - val_loss: 0.0048
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0039 - val_loss: 0.0043
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0034 - val_loss: 0.0038
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0032 - val_loss: 0.0048
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 39s 12ms/step - loss: 0.0031 - val_loss: 0.0039
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 39s 12ms/step - loss: 0.0030 - val_loss: 0.0040
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0030 - val_loss: 0.0037
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0029 - val_loss: 0.0035
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0029 - val_loss: 0.0035
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0028 - val_loss: 0.0035


In [45]:
rnn_rmsprop_result = evaluate_model(
    rnn_rmsprop,
    X_test,
    y_test,
    scaler,
    168,
    'RNN + RMSprop'
)

In [46]:
comparison_rnn = pd.DataFrame([
    baseline_rnn_result,
    rnn_early_result,
    rnn_dropout_result,
    rnn_more_units_result,
    rnn_batchnorm_result,
    rnn_rmsprop_result
])

comparison_rnn = comparison_rnn.sort_values('RMSE').reset_index(drop=True)

comparison_rnn

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + RMSprop,1395.841119,3.787807e+06,1946.229041,4.478693,0.905124,226.448594
1,168,RNN + EarlyStopping,1404.789758,3.805389e+06,1950.740600,4.476750,0.904684,70.623907
2,168,Baseline RNN (Multivariate),1420.956207,3.852737e+06,1962.838902,4.561250,0.903498,127.218473
3,168,RNN + More Units,1640.796760,4.978140e+06,2231.174637,5.199900,0.875309,-435.698344
4,168,RNN + Dropout,1709.290569,5.318545e+06,2306.197167,5.450563,0.866782,-181.105101
5,168,RNN + BatchNormalization,3246.835600,1.624687e+07,4030.740491,10.685205,0.593052,81.122771


In [47]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

rnn_lr = Sequential([
    SimpleRNN(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

rnn_lr.compile(
    optimizer='adam',
    loss='mse'
)

rnn_lr_history = rnn_lr.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)



/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 43s 13ms/step - loss: 0.0081 - val_loss: 0.0047 - learning_rate: 0.0010
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 39s 12ms/step - loss: 0.0038 - val_loss: 0.0046 - learning_rate: 0.0010
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 39s 12ms/step - loss: 0.0033 - val_loss: 0.0040 - learning_rate: 0.0010
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 39s 12ms/step - loss: 0.0031 - val_loss: 0.0039 - learning_rate: 0.0010
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0030 - val_loss: 0.0036 - learning_rate: 0.0010
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0029 - val_loss: 0.0036 - learning_rate: 0.0010
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0028 - val_loss: 0.0036 - learning_rate: 0.0010
Epoch 8/10
3168/3171 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0028
Epoch 8: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.0

In [48]:
rnn_lr_result = evaluate_model(
    rnn_lr,
    X_test,
    y_test,
    scaler,
    168,
    'RNN + LR Scheduler'
)

In [49]:
comparison_rnn = pd.DataFrame([
    baseline_rnn_result,
    rnn_early_result,
    rnn_dropout_result,
    rnn_more_units_result,
    rnn_batchnorm_result,
    rnn_rmsprop_result,
    rnn_lr_result
])

comparison_rnn = comparison_rnn.sort_values('RMSE').reset_index(drop=True)

comparison_rnn

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + LR Scheduler,1294.145288,3.444886e+06,1856.040518,4.119253,0.913713,91.140946
1,168,RNN + RMSprop,1395.841119,3.787807e+06,1946.229041,4.478693,0.905124,226.448594
2,168,RNN + EarlyStopping,1404.789758,3.805389e+06,1950.740600,4.476750,0.904684,70.623907
3,168,Baseline RNN (Multivariate),1420.956207,3.852737e+06,1962.838902,4.561250,0.903498,127.218473
4,168,RNN + More Units,1640.796760,4.978140e+06,2231.174637,5.199900,0.875309,-435.698344
5,168,RNN + Dropout,1709.290569,5.318545e+06,2306.197167,5.450563,0.866782,-181.105101
6,168,RNN + BatchNormalization,3246.835600,1.624687e+07,4030.740491,10.685205,0.593052,81.122771


# rnn + Batch Size =16

In [50]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_bs16 = Sequential([
    SimpleRNN(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

rnn_bs16.compile(
    optimizer='adam',
    loss='mse'
)

rnn_bs16_history = rnn_bs16.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=16
)

rnn_bs16_result = evaluate_model(
    rnn_bs16,
    X_test,
    y_test,
    scaler,
    168,
    'RNN + Batch Size 16'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 70s 11ms/step - loss: 0.0064 - val_loss: 0.0046
Epoch 2/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 64s 10ms/step - loss: 0.0035 - val_loss: 0.0038
Epoch 3/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 63s 10ms/step - loss: 0.0032 - val_loss: 0.0036
Epoch 4/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 64s 10ms/step - loss: 0.0030 - val_loss: 0.0035
Epoch 5/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 64s 10ms/step - loss: 0.0030 - val_loss: 0.0034
Epoch 6/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 65s 10ms/step - loss: 0.0029 - val_loss: 0.0035
Epoch 7/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 65s 10ms/step - loss: 0.0029 - val_loss: 0.0034
Epoch 8/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 63s 10ms/step - loss: 0.0029 - val_loss: 0.0033
Epoch 9/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 64s 10ms/step - loss: 0.0028 - val_loss: 0.0033
Epoch 10/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 64s 10ms/step - loss: 0.0028 - val_loss: 0.0034


# rnn + Batch Size =64

In [51]:
rnn_bs64 = Sequential([
    SimpleRNN(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

rnn_bs64.compile(
    optimizer='adam',
    loss='mse'
)

rnn_bs64_history = rnn_bs64.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

rnn_bs64_result = evaluate_model(
    rnn_bs64,
    X_test,
    y_test,
    scaler,
    168,
    'RNN + Batch Size 64'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - loss: 0.0101 - val_loss: 0.0051
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0043 - val_loss: 0.0044
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0036 - val_loss: 0.0041
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0033 - val_loss: 0.0042
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0032 - val_loss: 0.0039
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0031 - val_loss: 0.0038
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0030 - val_loss: 0.0040
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0029 - val_loss: 0.0039
Epoch 9/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0029 - val_loss: 0.0036
Epoch 10/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0029 - val_loss: 0.0035


In [52]:
batchsize_comparison = pd.DataFrame([
    baseline_rnn_result,
    rnn_bs16_result,
    rnn_bs64_result
])

batchsize_comparison = batchsize_comparison.sort_values('RMSE').reset_index(drop=True)

batchsize_comparison

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + Batch Size 64,1369.695706,3.646054e+06,1909.464414,4.331280,0.908675,-226.219111
1,168,RNN + Batch Size 16,1373.702789,3.681289e+06,1918.668442,4.419739,0.907792,265.748111
2,168,Baseline RNN (Multivariate),1420.956207,3.852737e+06,1962.838902,4.561250,0.903498,127.218473


# Add extra layers

In [53]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_two_layers = Sequential([
    SimpleRNN(64, return_sequences=True, input_shape=(sequence_length, X_train.shape[2])),
    SimpleRNN(32),
    Dense(24)
])

rnn_two_layers.compile(
    optimizer='adam',
    loss='mse'
)

rnn_two_layers.summary()

rnn_two_layers_history = rnn_two_layers.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_13 (SimpleRNN)       │ (None, 168, 64)        │         5,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_14 (SimpleRNN)       │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 24)             │           792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,952 (34.97 KB)

 Trainable params: 8,952 (34.97 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 78s 24ms/step - loss: 0.0084 - val_loss: 0.0054
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 72s 23ms/step - loss: 0.0037 - val_loss: 0.0039
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 71s 22ms/step - loss: 0.0033 - val_loss: 0.0038
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 71s 22ms/step - loss: 0.0031 - val_loss: 0.0037
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 69s 22ms/step - loss: 0.0030 - val_loss: 0.0036
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 70s 22ms/step - loss: 0.0028 - val_loss: 0.0036
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 71s 22ms/step - loss: 0.0028 - val_loss: 0.0035
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 71s 22ms/step - loss: 0.0028 - val_loss: 0.0033
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 69s 22ms/step - loss: 0.0029 - val_loss: 0.0034
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 70s 22ms/step - loss: 0.0027 - val_loss: 0.0035


In [54]:
rnn_two_layers_result = evaluate_model(
    rnn_two_layers,
    X_test,
    y_test,
    scaler,
    168,
    'RNN + Two Layers'
)

pd.DataFrame([rnn_two_layers_result])

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + Two Layers,1443.613954,3.910746e+06,1977.560513,4.683462,0.902045,515.13167


In [55]:
comparison_rnn = pd.DataFrame([
    baseline_rnn_result,
    rnn_early_result,
    rnn_dropout_result,
    rnn_more_units_result,
    rnn_batchnorm_result,
    rnn_rmsprop_result,
    rnn_lr_result,
    rnn_bs16_result,
    rnn_bs64_result,
    rnn_two_layers_result
])

comparison_rnn = comparison_rnn.sort_values('RMSE').reset_index(drop=True)

comparison_rnn

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + LR Scheduler,1294.145288,3.444886e+06,1856.040518,4.119253,0.913713,91.140946
1,168,RNN + Batch Size 64,1369.695706,3.646054e+06,1909.464414,4.331280,0.908675,-226.219111
2,168,RNN + Batch Size 16,1373.702789,3.681289e+06,1918.668442,4.419739,0.907792,265.748111
3,168,RNN + RMSprop,1395.841119,3.787807e+06,1946.229041,4.478693,0.905124,226.448594
4,168,RNN + EarlyStopping,1404.789758,3.805389e+06,1950.740600,4.476750,0.904684,70.623907
5,168,Baseline RNN (Multivariate),1420.956207,3.852737e+06,1962.838902,4.561250,0.903498,127.218473
6,168,RNN + Two Layers,1443.613954,3.910746e+06,1977.560513,4.683462,0.902045,515.131670
7,168,RNN + More Units,1640.796760,4.978140e+06,2231.174637,5.199900,0.875309,-435.698344
8,168,RNN + Dropout,1709.290569,5.318545e+06,2306.197167,5.450563,0.866782,-181.105101
9,168,RNN + BatchNormalization,3246.835600,1.624687e+07,4030.740491,10.685205,0.593052,81.122771


In [56]:
comparison_rnn.to_csv('/kaggle/working/Comparison_csv')

In [63]:
import keras_tuner as kt
# ---------------------------------------------------
# Build RNN model for hyperparameter tuning
# ---------------------------------------------------
def build_rnn(hp):
    model = Sequential()
    model.add(
        SimpleRNN(
            units=hp.Choice(
                'rnn_units',
                values=[32, 64, 128]
            ),
            input_shape=(sequence_length, X_train.shape[2])
        )
    )

    model.add(
        Dropout(
            hp.Choice(
                'dropout_rate',
                values=[0.0, 0.2, 0.3]
            )
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                'dense_units',
                values=[32, 64, 128]
            ),
            activation='relu'
        )
    )

    model.add(Dense(24))

    learning_rate = hp.Choice(
        'learning_rate',
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss='mse',
        metrics=['mae']
    )

    return model


# ---------------------------------------------------
# Hyperparameter search
# ---------------------------------------------------
tuner = kt.RandomSearch(
    build_rnn,
    objective='val_loss',
    max_trials=3,
    executions_per_trial=1,
    directory='hyperparameter_tuning',
    project_name='rnn_168_to_24'
)

tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


# ---------------------------------------------------
# Get best hyperparameters
# ---------------------------------------------------
best_hp = tuner.get_best_hyperparameters(1)[0]

print('Best RNN Units:', best_hp.get('rnn_units'))
print('Best Dropout:', best_hp.get('dropout_rate'))
print('Best Dense Units:', best_hp.get('dense_units'))
print('Best Learning Rate:', best_hp.get('learning_rate'))


# ---------------------------------------------------
# Rebuild and retrain the best model
# ---------------------------------------------------

final_rnn = build_rnn(best_hp)

final_rnn_history = final_rnn.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,              
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# ---------------------------------------------------
# Evaluate the retrained final model
# ---------------------------------------------------

final_rnn_result = evaluate_model(
    final_rnn,
    X_test,
    y_test,
    scaler,
    168,
    'Final Tuned RNN'
)

pd.DataFrame([final_rnn_result])

# ---------------------------------------------------
# Save the final tuned model
# ---------------------------------------------------

final_rnn.save('/kaggle/working/rnn168_phase6_final_tuned.keras')

Trial 3 Complete [00h 03m 27s]
val_loss: 0.0035219599958509207

Best val_loss So Far: 0.0035219599958509207
Total elapsed time: 00h 10m 26s
Best RNN Units: 128
Best Dropout: 0.0
Best Dense Units: 128
Best Learning Rate: 0.001
Epoch 1/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 44s 13ms/step - loss: 0.0051 - mae: 0.0511 - val_loss: 0.0038 - val_mae: 0.0441
Epoch 2/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 40s 13ms/step - loss: 0.0033 - mae: 0.0419 - val_loss: 0.0036 - val_mae: 0.0429
Epoch 3/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 40s 13ms/step - loss: 0.0030 - mae: 0.0398 - val_loss: 0.0036 - val_mae: 0.0432
Epoch 4/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 40s 13ms/step - loss: 0.0029 - mae: 0.0387 - val_loss: 0.0034 - val_mae: 0.0420
Epoch 5/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 40s 12ms/step - loss: 0.0028 - mae: 0.0380 - val_loss: 0.0034 - val_mae: 0.0420
Epoch 6/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 40s 12ms/step - loss: 0.0027 - mae: 0.0373 - val_loss: 0.0032 - val_mae: 0.0403
Epoch 7/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 

In [64]:
final_rnn_result

{'Sequence Length': 168,
 'Model': 'Final Tuned RNN',
 'MAE': 1340.0381780405485,
 'MSE': 3456849.4009657144,
 'RMSE': np.float64(1859.2604446299918),
 'MAPE': 4.314541041179043,
 'R2': 0.9134136861252531,
 'Bias': np.float64(167.3316408985709)}

In [65]:
comparison_rnn = pd.DataFrame([
    baseline_rnn_result,
    rnn_early_result,
    rnn_dropout_result,
    rnn_more_units_result,
    rnn_batchnorm_result,
    rnn_rmsprop_result,
    rnn_lr_result,
    rnn_bs16_result,
    rnn_bs64_result,
    rnn_two_layers_result,
    final_rnn_result
])

comparison_rnn = comparison_rnn.sort_values('RMSE').reset_index(drop=True)

comparison_rnn

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,RNN + LR Scheduler,1294.145288,3.444886e+06,1856.040518,4.119253,0.913713,91.140946
1,168,Final Tuned RNN,1340.038178,3.456849e+06,1859.260445,4.314541,0.913414,167.331641
2,168,RNN + Batch Size 64,1369.695706,3.646054e+06,1909.464414,4.331280,0.908675,-226.219111
3,168,RNN + Batch Size 16,1373.702789,3.681289e+06,1918.668442,4.419739,0.907792,265.748111
4,168,RNN + RMSprop,1395.841119,3.787807e+06,1946.229041,4.478693,0.905124,226.448594
5,168,RNN + EarlyStopping,1404.789758,3.805389e+06,1950.740600,4.476750,0.904684,70.623907
6,168,Baseline RNN (Multivariate),1420.956207,3.852737e+06,1962.838902,4.561250,0.903498,127.218473
7,168,RNN + Two Layers,1443.613954,3.910746e+06,1977.560513,4.683462,0.902045,515.131670
8,168,RNN + More Units,1640.796760,4.978140e+06,2231.174637,5.199900,0.875309,-435.698344
9,168,RNN + Dropout,1709.290569,5.318545e+06,2306.197167,5.450563,0.866782,-181.105101
